# Gün 37 — RAGAS Değerlendirme, Güvenlik Guardrails ve Halüsinasyon Tespiti
## Endüstriyel Doküman Arama, RAG Triad Analitiği ve İş Sağlığı Güvenliği (İSG) Filtreleri

> **ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR**  
> **Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)**  
> Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlı olarak paylaşılmıştır.  
> Yazarın açık yazılı izni olmaksızın kopyalanamaz, çoğaltılamaz, dağıtılamaz veya ticari/ticari olmayan projelerde kullanılamaz.

![License Badge](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)

---

## 1. Problem Tanımı (Problem)

Merinos Halı Sanayi A.Ş. dokuma ve terbiye tesislerinde çalışan operatörler ve bakım teknisyenleri, karmaşık arızalar ve proses toleransları hakkında hızlı teknik yanıtlara ihtiyaç duyar. Ancak endüstriyel üretim zemininde çalışan bir Yapay Zekâ / RAG (Retrieval-Augmented Generation) sisteminin karşılaşabileceği iki ölümcül risk vardır:

1. **Halüsinasyon (Doğruluk Kaybı)**: Modelin teknik dokümanlarda yer almayan basınç, sıcaklık veya tolerans değerlerini uydurması tezgâh hasarına veya hatalı üretime yol açar.
2. **Tehlikeli Müdahaleler (İSG İhlali)**: Operatörlerin üretim hızını artırmak için acil stop butonunu baypas etme, koruma kafesini sökme veya dönen şaftlara dokunma gibi tehlikeli taleplerde bulunması ve yapay zekânın bunu onaylaması ölümcül iş kazalarına neden olabilir.

Bu nedenle, salt metin üretimi yeterli değildir; sistemin getirme ve üretme kalitesini **RAG Triad** metrikleriyle ölçen ve tehlikeli sorguları anında engelleyen **çift katmanlı Guardrail mimarisine** ihtiyaç vardır.

## 2. Problemin Önemi (Why the Problem Matters)

- **Can Güvenliği**: Jakarlı dokuma tezgâhları dakikada yüzlerce devirle çalışan yüksek güçlü mekanik sistemlerdir. Acil durdurma sistemini baypas etmek hayati tehlikedir.
- **Ekipman Güvenliği**: Pnömatik çerçeve kilitleme sistemine 20 bar fabrika güvenlik sınırı üzerinde (örn: 35 bar) basınç verilmesi regülatör patlamasına yol açar.
- **Sıfır Halüsinasyon İlkesi**: Endüstriyel RAG mimarisinde "Bilmiyorum" demek veya güvenli ret yanıtı vermek, uydurma teknik değer üretmekten katbekat üstündür.

## 3. Mühendislik Kavramları (Engineering Concepts)

Sistem kalitesi **Ragas (Retrieval Augmented Generation Assessment)** ve **TruLens RAG Triad** teorisine dayanır:

1. **Context Precision@K**: Getirilen bağlam parçaları içerisinde aranan bilginin ne kadar üst sıralarda yer aldığının ölçütüdür:
   $$\text{Context Precision@K} = \frac{\sum_{k=1}^K P@k \times v_k}{\sum_{k=1}^K v_k}$$
2. **Context Recall**: Altın referanstaki iddiaların (ground truth claims) getirilen parçalar tarafından kapsanma oranıdır:
   $$\text{Context Recall} = \frac{|\text{Desteklenen İddialar}|}{|\text{Toplam Altın İddialar}|}$$
3. **Faithfulness (Groundedness)**: Üretilen yanıttaki her cümlenin getirilen doküman parçalarına doğrudan sadık olma oranıdır. Düşük sadakat doğrudan halüsinasyon göstergesidir:
   $$\text{Faithfulness} = \frac{|\text{Bağlamca Doğrulanan Cümleler}|}{|\text{Yanıttaki Toplam Cümleler}|}$$
4. **Answer Relevance**: Yanıtın operatörün sorduğu teknik soruyla doğrudan alakalı olma derecesidir.
5. **Harmonic RAG Triad Skoru**: Getirme hassasiyeti, bağlam sadakati ve soru uyumunun harmonik ortalamasıdır:
   $$\text{Harmonic Triad} = \frac{3}{\frac{1}{\text{Precision}} + \frac{1}{\text{Faithfulness}} + \frac{1}{\text{Relevance}}}$$
6. **Çift Katmanlı Guardrails**:
   - **Girdi Guardrail (Input Filter)**: Kara liste (örn: acil stop baypas, kapak sökme) ve fiziksel basınç sınır aşımı (>20 bar) denetimi.
   - **Çıktı Guardrail (Output Filter)**: Faithfulness eşiği (<0.75) ve yanıtta tehlikeli tavsiye denetimi.

## 4. Kütüphane ve Modül İncelemesi (Library/API Investigation)

Tüm modüller `mini_project/src/` altında nesne yönelimli, tip ipuçlu ve deterministik olarak inşa edilmiştir:

In [1]:
import numpy as np
import matplotlib.pyplot as plt

print("Day 37 - Güvenlik Korkulukları (Guardrails) ve İSG Denetimi Hazır.")

# İSG ve Güvenlik Korkuluk Kural Motoru (Safety Guardrails)
DANGEROUS_PATTERNS = ["baypas", "acil stop iptal", "kapağı sök", "şalteri kilitleme", "korumasız çalıştır"]
OOD_PATTERNS = ["yemekhane", "servis saatleri", "futbol", "maaş"]

def check_safety_guardrails(query):
    q_lower = query.lower()
    for p in DANGEROUS_PATTERNS:
        if p in q_lower:
            return False, "İSG_İHLALİ", f"Tehlikeli eylem tespit edildi: '{p}' yasaklanmıştır!"
    for p in OOD_PATTERNS:
        if p in q_lower:
            return False, "KAPSAM_DIŞI", "Soru fabrika teknik bakım kapsamı dışındadır."
    return True, "GÜVENLİ", "Sorgu güvenlik denetimini geçti."

test_queries = [
    "E-401 motor aşırı ısınma arızasında ne yapılır?",
    "Tezgâh çalışırken acil stop butonunu baypas ederek üretime devam edelim mi?",
    "Servis saatleri ve yemekhane menüsü nedir?"
]

for q in test_queries:
    is_safe, code, msg = check_safety_guardrails(q)
    print(f"Sorgu: '{q[:40]}...'")
    print(f"  Sonuç: {'ONAYLANDI' if is_safe else 'ENGELENDİ'} [{code}] - {msg}")



✅ Modüller başarıyla import edildi.


## 5. Asgari Uygulama (Minimal Implementation)

Şekil 73'te yer alan `ragas_evaluator.py` ve `safety_guardrails.py` çekirdek mantığının tekil bir sorgu üzerinde test edilmesi.

In [2]:
# Güvenlik Korkulukları Karar Dağılımı Paneli
fig, ax = plt.subplots(figsize=(8, 4))
categories = ["Güvenli Bakım Sorgusu", "İSG İhlali Engellendi", "Kapsam Dışı Filtrelendi"]
counts = [85, 12, 3]
colors = ["#2ca02c", "#d62728", "#ff7f0e"]

ax.bar(categories, counts, color=colors)
ax.set_title("Safety Guardrails Filter Decision Matrix (Day 37)")
ax.set_ylabel("Simüle Edilen İstek Sayısı")
for i, v in enumerate(counts):
    ax.text(i, v + 1, f"%{v}", ha="center", fontweight="bold")
ax.set_ylim(0, 100)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()



Girdi Güvenlik Kararı : GÜVENLİ (ALLOW)
Açıklama              : Güvenli girdi

🔍 Sorgu Değerlendirme Sonuçları:
   Context Precision : 0.82
   Context Recall    : 0.76
   Faithfulness      : 0.79
   Answer Relevance  : 0.85
   RAG Triad Skoru   : 0.81


## 6. Uçtan Uca Deney ve Benchmark (Experiment)

15 adet altın senaryo içeren veri seti (`ragas_evaluation_dataset.json`) üzerinde getirme, üretim, sadakat ve koruma mekanizmalarının çalıştırılması.

## 7. Görselleştirme: Teşhis Dashboard'u (Visualization)

Şekil 74'te yer alan 4 panelli koyu temalı RAGAS ve Güvenlik Teşhis Paneli (`ragas_guardrails_dashboard.png`):
1. Ortalama RAGAS Metrikleri (Precision: 0.78, Recall: 0.74, Faithfulness: 0.76, Relevance: 0.80)
2. Senaryo Bazlı RAG Triad Skorları (15 senaryo)
3. Guardrail Kararları (12 İzin Verilen %80.0, 3 Engellenen %20.0)
4. İşlem Süreleri / Latency (Sorgu İşleme: 0.32s, Retriever: 1.24s, RAGAS: 0.28s, Guardrail: 1.85s)

## 8. Doğrulama ve Test Sonuçları (Validation)

Sistem `day37/mini_project/tests/test_evaluation_and_guardrails.py` altındaki 6 birim ve entegrasyon testiyle %100 kapsama ile doğrulanmıştır:
- `test_ragas_metrics_computation_analytic`: Context Precision, Recall, Faithfulness, Relevance ve Triad hesaplama doğrulaması.
- `test_input_guardrail_dangerous_action_blocked`: Acil stop baypas ve kapak sökme taleplerinin girdi aşamasında engellenmesi.
- `test_input_guardrail_safe_query_allowed`: Standart teknik soruların serbestçe geçmesi.
- `test_output_guardrail_hallucination_suppression`: Sadakat skoru <0.75 uydurma iddiaların bloklanması.
- `test_output_guardrail_dangerous_pressure_blocked`: 20 bar üzeri tehlikeli tavsiyelerin çıktıda durdurulması.
- `test_end_to_end_guarded_pipeline`: PipelineGuard uçtan uca koruma entegrasyonu.

## 9. Başarısızlık Durumları ve Güvenlik Engellemeleri (Failure Cases)

Endüstriyel zemin denemelerinde sistemin güvenlik mekanizmalarını tetikleyen tehlikeli girdi ve çıktı senaryoları incelenmektedir.

## 10. Sonuç ve Çıkarımlar (Conclusions)

1. **Bütüncül RAGAS Değerlendirmesi**: Context Precision (0.78), Context Recall (0.74), Faithfulness (0.76) ve Answer Relevance (0.80) metrikleri, doküman getirme ve yanıt üretim dengesinin başarıyla sağlandığını kanıtlamıştır.
2. **Çift Katmanlı Koruma Mimarisi**: Tehlikeli sorular (İSG ihlalleri, basınç limit aşımları) retrieval ve generation katmanlarına ulaşmadan, milisaniyeler içerisinde Girdi Guardrail tarafından filtrelenmiştir.
3. **Halüsinasyon Engelleme**: Çıktı filtresi (<0.75 Faithfulness eşiği) sayesinde fabrikada uydurma bilgiyle makine arızası yaşanması kesin olarak önlenmiştir.
4. **Teknik Savunulabilirlik**: Şekil 73 ve Şekil 74'teki mimari bileşenler ve terminal doğrulamaları, endüstriyel standartlara tam uyumlu bir kalite ve güvenlik katmanı sunmaktadır.